In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
file_path = "/content/sample_data/transactions_train.csv"  # change path if needed
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/transactions_train.csv'

In [ ]:
df = df.copy()

# Convert datetime
df["transaction_time"] = pd.to_datetime(df["transaction_time"])

# Extract time features
df["txn_hour"] = df["transaction_time"].dt.hour
df["txn_day"] = df["transaction_time"].dt.day
df["txn_month"] = df["transaction_time"].dt.month
df["txn_dayofweek"] = df["transaction_time"].dt.dayofweek

# Drop raw timestamp
df.drop(columns=["transaction_time"], inplace=True)

In [ ]:
# IDs usually cause leakage or noise
df.drop(columns=["transaction_id", "customer_id", "merchant_id"], inplace=True)

In [ ]:
categorical_cols = ["payment_channel", "device_type"]

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
target = "is_fraud"

X = df.drop(columns=[target])
y = df[target]

print("Feature shape:", X.shape)

Feature shape: (221648, 23)


In [ ]:
non_nan_mask = ~y.isna()
X_filtered = X[non_nan_mask]
y_filtered = y[non_nan_mask]

X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, y_filtered,
    test_size=0.2,
    random_state=42,
    stratify=y_filtered
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (177317, 23)
Test shape: (44330, 23)


In [ ]:
scaler = StandardScaler()

numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns

# Fit ONLY on train
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# Transform test using same scaler
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [ ]:
train_processed = pd.concat([X_train, y_train], axis=1)
test_processed = pd.concat([X_test, y_test], axis=1)

train_processed.to_csv("transactions_train_processed.csv", index=False)
test_processed.to_csv("transactions_test_processed.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

In [ ]:
df = pd.read_csv("transactions_train_processed.csv")

target = "is_fraud"

X = df.drop(columns=[target]).values
y = df[target].values

# Convert X to a float type NumPy array to handle potential object/boolean columns
X = X.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class FraudDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = FraudDataset(X_train, y_train)
test_dataset = FraudDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512)

In [ ]:
class FraudCNN(nn.Module):
    def __init__(self, num_features):
        super(FraudCNN, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        # reshape to (batch, channels=1, features)
        x = x.unsqueeze(1)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))

        x = self.global_pool(x).squeeze(-1)
        x = self.dropout(x)

        x = self.fc(x)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_features = X_train.shape[1]
model = FraudCNN(num_features).to(device)

pos_weight = torch.tensor([ (len(y_train) - y_train.sum()) / y_train.sum() ]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch).squeeze()
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

Epoch [1/10], Loss: 0.4535
Epoch [2/10], Loss: 0.2116
Epoch [3/10], Loss: 0.1715
Epoch [4/10], Loss: 0.1537
Epoch [5/10], Loss: 0.1529
Epoch [6/10], Loss: 0.1376
Epoch [7/10], Loss: 0.1383
Epoch [8/10], Loss: 0.1301
Epoch [9/10], Loss: 0.1269
Epoch [10/10], Loss: 0.1211


In [ ]:
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch).squeeze()
        probs = torch.sigmoid(outputs)

        all_preds.extend(probs.cpu().numpy())
        all_targets.extend(y_batch.numpy())

roc = roc_auc_score(all_targets, all_preds)
print("ROC-AUC:", roc)

pred_labels = (np.array(all_preds) > 0.5).astype(int)
print(classification_report(all_targets, pred_labels))

ROC-AUC: 0.9984192659616437
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     34924
         1.0       0.84      0.94      0.89       540

    accuracy                           1.00     35464
   macro avg       0.92      0.97      0.94     35464
weighted avg       1.00      1.00      1.00     35464



In [ ]:
# Save only the model weights (recommended)
model_path = "fraud_cnn_model.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "num_features": num_features
}, model_path)

print(f"Model saved to {model_path}")

Model saved to fraud_cnn_model.pth


In [65]:
# Recreate model architecture first
checkpoint = torch.load("fraud_cnn_model.pth", map_location=device)

loaded_model = FraudCNN(checkpoint["num_features"]).to(device)
loaded_model.load_state_dict(checkpoint["model_state_dict"])

loaded_model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [ ]:
import os
import traceback
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib

from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel
from typing import List

# =====================================================
# CONFIG
# =====================================================

BASE_DIR = os.path.dirname(os.path.abspath(__file__))

MODEL_PATH = os.path.join(BASE_DIR, "fraud_cnn_model.pth")
TRAIN_PROCESSED_PATH = os.path.join(BASE_DIR, "transactions_train_processed.csv")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

app = FastAPI(title="Fraud Detection API", version="3.0")

# =====================================================
# MODEL DEFINITION
# =====================================================

class FraudCNN(nn.Module):
    def __init__(self, num_features):
        super(FraudCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(64)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)

        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128, 1)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

# =====================================================
# LOAD MODEL
# =====================================================

if not os.path.exists(MODEL_PATH):
    raise RuntimeError(f"Model not found at {MODEL_PATH}")

checkpoint = torch.load(MODEL_PATH, map_location=device)

num_features = checkpoint["num_features"]

model = FraudCNN(num_features).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# =====================================================
# RECOVER TRAINING COLUMN ORDER
# =====================================================

if not os.path.exists(TRAIN_PROCESSED_PATH):
    raise RuntimeError("transactions_train_processed.csv not found")

train_df = pd.read_csv(TRAIN_PROCESSED_PATH)

# Drop target
if "is_fraud" in train_df.columns:
    train_df.drop(columns=["is_fraud"], inplace=True)

training_columns = train_df.columns.tolist()

# =====================================================
# PREPROCESSING
# =====================================================

def preprocess_input(df: pd.DataFrame):
    df = df.copy()

    # Always drop target if exists
    if "is_fraud" in df.columns:
        df.drop(columns=["is_fraud"], inplace=True)

    # Datetime
    if "transaction_time" in df.columns:
        df["transaction_time"] = pd.to_datetime(
            df["transaction_time"], errors="coerce"
        )
        df["txn_hour"] = df["transaction_time"].dt.hour
        df["txn_day"] = df["transaction_time"].dt.day
        df["txn_month"] = df["transaction_time"].dt.month
        df["txn_dayofweek"] = df["transaction_time"].dt.dayofweek
        df.drop(columns=["transaction_time"], inplace=True)

    # Drop IDs
    drop_cols = ["transaction_id", "customer_id", "merchant_id"]
    df.drop(columns=[c for c in drop_cols if c in df.columns],
            errors="ignore",
            inplace=True)

    # One-hot
    categorical_cols = ["payment_channel", "device_type"]
    df = pd.get_dummies(
        df,
        columns=[c for c in categorical_cols if c in df.columns],
        drop_first=True
    )

    # Align strictly
    df = df.reindex(columns=training_columns, fill_value=0)

    return df

# =====================================================
# FILE PREDICTION
# =====================================================

@app.post("/predict-file")
async def predict_file(file: UploadFile = File(...)):
    try:
        if file.filename.endswith(".csv"):
            df = pd.read_csv(file.file)
        elif file.filename.endswith(".xlsx"):
            df = pd.read_excel(file.file)
        else:
            return JSONResponse(
                {"error": "Only CSV or Excel files supported"},
                status_code=400
            )

        original_df = df.copy()

        df_processed = preprocess_input(df)

        if df_processed.shape[1] != num_features:
            return JSONResponse(
                {"error": f"Feature mismatch. Expected {num_features}, got {df_processed.shape[1]}"},
                status_code=400
            )

        X = torch.tensor(df_processed.values.astype(np.float32)).to(device)

        with torch.no_grad():
            outputs = model(X).squeeze()
            probs = torch.sigmoid(outputs).cpu().numpy()

        original_df["fraud_probability"] = probs
        original_df["is_fraud"] = probs > 0.5

        fraud_only = original_df[original_df["is_fraud"] == True]

        output_path = os.path.join(BASE_DIR, "detected_fraud.csv")
        fraud_only.to_csv(output_path, index=False)

        return FileResponse(
            output_path,
            media_type="text/csv",
            filename="detected_fraud.csv"
        )

    except Exception as e:
        traceback.print_exc()
        return JSONResponse({"error": str(e)}, status_code=500)

# =====================================================
# HEALTH CHECK
# =====================================================

@app.get("/")
def health():
    return {
        "status": "Running",
        "expected_features": num_features
    }

ROC-AUC: 0.9979559983121344
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00     43656
         1.0       0.85      0.92      0.89       674

    accuracy                           1.00     44330
   macro avg       0.93      0.96      0.94     44330
weighted avg       1.00      1.00      1.00     44330

